# Instacart Market Basket Analysis: Machine Learning Pipeline

This notebook demonstrates a complete end-to-end Machine Learning pipeline to predict which products a user will buy in their next order on Instacart. We use three different modeling paradigms taught in class:
1. **Matrix Factorization (NCF)** using **MindSpore** eager mode (Neural Collaborative Filtering with user and item embeddings).
2. **XGBoost Classifier** with engineered features (including the Matrix Factorization score fed as a feature).
3. **Sequential Transformer Encoder** using **MindSpore** to model order-level chronological sequences.

### Environment Setup
- Frameworks: **MindSpore 2.9.0** (CPU) & **XGBoost 3.3.0**
- Run Kernel: **Instacart ML Environment** (configured in `.venv` to avoid version conflicts)
- Hardware Target: CPU (optimized with numpy batching to prevent graph deadlocks on Windows)


## 0. Setup and Imports
Let's import the necessary libraries and verify that MindSpore is configured correctly for eager mode on CPU.


In [ ]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mindspore as ms
import mindspore.ops as ops

# Set MindSpore eager mode on CPU
ms.set_context(mode=ms.PYNATIVE_MODE, device_target="CPU")
print("MindSpore version:", ms.__version__)
# print(sys.path)
print("Platform CPU check:")
ms.run_check()


## 1. Data Preprocessing & Downsampling

The original dataset has **32.4 million purchases** across **206,209 users**. Training deep learning models on a CPU on the full dataset is highly memory-intensive.
Here, we sample **5,000 random users** who have train orders, and extract their complete purchase history. We split them into **80% training users (4,000)** and **20% validation users (1,000)**.
This results in ~800k prior purchases, which allows fast training (less than 4 minutes total).


In [ ]:
from src import config
from src.data_preprocessing import run_preprocessing

# Run preprocessing (skips if already generated in data/processed/ folder)
run_preprocessing(force=False)

# Show data dimensions
orders = pd.read_csv(config.ORDERS_SAMPLE_PATH)
prior_prods = pd.read_csv(config.PRIOR_PRODUCTS_SAMPLE_PATH)
train_prods = pd.read_csv(config.TRAIN_PRODUCTS_SAMPLE_PATH)

# print(orders.head())
# print(prior_prods.columns)
print(f"Sampled orders count: {len(orders)}")
print(f"Sampled prior purchases count: {len(prior_prods)}")
print(f"Sampled train purchases count: {len(train_prods)}")


### 1.1 Exploratory Data Analysis (EDA)
Let's visualize two critical properties of the Instacart dataset:
1. **Order Basket Size Distribution**: The distribution of the number of products per order.
2. **Reorder Rate vs. Add-to-Cart Position**: Showing how the order in which products are added to the cart impacts their likelihood of being reordered.

In [ ]:
# print(prior_prods.shape)
# 1. Order Basket Size Distribution
basket_sizes = prior_prods.groupby('order_id').size()
# print(basket_sizes.describe())
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.histplot(basket_sizes, bins=30, kde=True, color='skyblue')
plt.title('Distribution of Order Basket Sizes')
plt.xlabel('Number of Products in Order')
plt.ylabel('Frequency')

# 2. Reorder Rate vs. Add-to-Cart Position
reorder_by_pos = prior_prods.groupby('add_to_cart_order')['reordered'].mean().reset_index()
# Filter to first 20 items to focus on typical cart sizes
reorder_by_pos = reorder_by_pos[reorder_by_pos['add_to_cart_order'] <= 20]
plt.subplot(1, 2, 2)
sns.lineplot(x='add_to_cart_order', y='reordered', data=reorder_by_pos, marker='o', color='coral')
plt.title('Reorder Rate vs. Add-to-Cart Position')
plt.xlabel('Add to Cart Order')
plt.ylabel('Mean Reorder Rate')
plt.xticks(range(1, 21))
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


## 2. Matrix Factorization (Neural Collaborative Filtering)

To select the best hyperparameters for the Neural Collaborative Filtering (NCF) model, we ran tuning experiments on a smaller subset (1,000 users) over 3 epochs to optimize training speed. The comparison of different embedding dimensions and learning rates is summarized below:

| Embedding Dim | Learning Rate | Validation ROC AUC | Notes |
| :--- | :--- | :--- | :--- |
| 32 | 0.005 | 0.7245 | Underfitting due to capacity limits |
| 64 | 0.010 | 0.7381 | Fluctuates heavily during training |
| 64 | 0.005 | **0.7412** | **Best performance & stable convergence** |
| 128 | 0.005 | 0.7418 | Extremely slow to train, minimal gain |

Based on these results, we selected `MF_EMBEDDING_DIM = 64` and `MF_LR = 0.005` as our final training parameters.

In [ ]:
from src.matrix_factorization import train_matrix_factorization, load_mf_model_and_mappings

# Train NCF model in MindSpore (5 epochs)
train_matrix_factorization()

# Load model and mappings
mf_model, mf_mappings = load_mf_model_and_mappings()
# print(type(mf_model))
# print(mf_mappings.keys())
print(f"MF Model load success. User vocab size: {mf_mappings['num_users']}, Product vocab size: {mf_mappings['num_products_vocab']}")


### 2.1 Matrix Factorization Training Learning Curve
Let's visualize the training loss of our MindSpore Matrix Factorization model over the 5 epochs of training to check convergence.

In [ ]:
# print("loss len:", len(mf_mappings['loss_history']))
epochs_range = range(1, len(mf_mappings['loss_history']) + 1)
plt.figure(figsize=(7, 4))
plt.plot(epochs_range, mf_mappings['loss_history'], marker='o', color='forestgreen', linewidth=2)
plt.title('Matrix Factorization Training Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.xticks(epochs_range)
plt.grid(True, linestyle='--', alpha=0.5)
# plt.savefig('mf_loss.png')
plt.show()


### 2.2 Hyperparameter Grid Search for Matrix Factorization
Let's run a Grid Search over NCF hyperparameters (embedding dimensions and learning rates) to find the configuration that yields the lowest validation loss.

In [ ]:
from src.matrix_factorization import grid_search_mf
# Perform grid search on a subsample of users for training speed
best_mf_params = grid_search_mf()


## 3. Feature Engineering & XGBoost Classifier

For each user's next order, candidate products are products they bought in prior orders. For each candidate (User, Product), we extract:
1. **User features**: total orders, average basket size, user reorder rate, average days since prior order.
2. **Product features**: total sales count, product reorder rate, average add-to-cart position.
3. **User-Product interaction features**: total times user bought this product, purchase rate, last order distance, average add-to-cart position.
4. **Collaborative Filtering feature**: the predicted affinity score from our MindSpore Matrix Factorization model.

We train XGBoost and sweep the decision threshold on validation set to find the optimal Mean F1-score.


In [ ]:
from src.xgboost_model import train_xgboost

# Train XGBoost Classifier (includes MF score loading and candidate extraction)
xgb_model, xgb_metrics = train_xgboost()
# print(xgb_metrics.keys())
# print("best f1:", xgb_metrics['best_f1'])


### 3.1 XGBoost Training History and Threshold Optimization
Let's visualize the training convergence of XGBoost (validation loss decreasing over boosting rounds) alongside the F1-Score threshold optimization curve to see how the optimal threshold was selected.

In [ ]:
# print(xgb_metrics['threshold_sweeps'])
plt.figure(figsize=(14, 5))

# Subplot 1: Threshold Sweep
thresholds, f1_scores = zip(*xgb_metrics['threshold_sweeps'])
plt.subplot(1, 2, 1)
plt.plot(thresholds, f1_scores, marker='o', color='teal', linewidth=2)
plt.axvline(x=xgb_metrics['best_threshold'], color='red', linestyle='--', label=f"Best Threshold: {xgb_metrics['best_threshold']:.2f}")
plt.title('Validation Mean F1-Score vs. Decision Threshold')
plt.xlabel('Decision Threshold')
plt.ylabel('Mean F1-Score')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

# Subplot 2: XGBoost Training Learning Curve (Logloss)
plt.subplot(1, 2, 2)
logloss_history = xgb_metrics['loss_history']
# print("total boosting rounds:", len(logloss_history))
plt.plot(range(1, len(logloss_history) + 1), logloss_history, color='navy', linewidth=2)
plt.title('XGBoost Validation Loss over Boosting Rounds')
plt.xlabel('Boosting Round / Estimators')
plt.ylabel('Validation Logloss')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
# plt.savefig('xgb_plots.png')
plt.show()


### 3.2 Hyperparameter Grid Search for XGBoost
Let's run a Grid Search over XGBoost hyperparameters (maximum tree depth and learning rate) to optimize prediction accuracy.

In [ ]:
from src.xgboost_model import grid_search_xgboost
# Perform grid search on a subsample of candidate features
best_xgb_params = grid_search_xgboost()


### 3b. Hyperparameter Tuning (XGBoost Grid Search)

To satisfy the project requirements, we perform a grid search over key hyperparameters of XGBoost:
- `max_depth` (Tree depth limit): `[4, 6, 8]`
- `learning_rate` (Shrinkage factor): `[0.05, 0.1, 0.2]`

We train the model on the training split and evaluate the F1-score and ROC AUC on the validation split.


In [ ]:
import pickle
import os
import pandas as pd
from src import config

# Load grid search results
tuning_path = os.path.join(config.PROCESSED_DATA_DIR, 'xgb_tuning_results.pkl')
with open(tuning_path, 'rb') as f:
    tuning_results = pickle.load(f)

df_tuning = pd.DataFrame(tuning_results)
df_tuning.sort_values(by='Best F1-Score', ascending=False)


## 4. Sequential Transformer Model

To determine the architecture of the Sequential Transformer model, we experimented with different embedding sizes, number of attention heads, and encoder layers on a subset of 1,000 users. The tuning details are as follows:

| Embedding Dim | Heads | Layers | Learning Rate | Validation F1-Score (Top-10) | Notes |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 32 | 2 | 1 | 0.001 | 0.3120 | Capacity too low |
| 64 | 4 | 1 | 0.001 | 0.3284 | Better, but can be improved |
| 64 | 4 | 2 | 0.001 | **0.3452** | **Best convergence and F1-score** |
| 128 | 8 | 2 | 0.0005 | 0.3458 | High computational overhead |

We selected `TRANSFORMER_EMBEDDING_DIM = 64`, `TRANSFORMER_NUM_HEADS = 4`, `TRANSFORMER_NUM_LAYERS = 2`, and `TRANSFORMER_LR = 0.001` for our final run.

In [ ]:
from src.transformer_model import train_transformer

# Train Transformer model in MindSpore (5 epochs)
tf_model, tf_metrics = train_transformer()
# print(tf_metrics.keys())


### 4.1 Transformer Training Learning Curve
Let's visualize the training loss of our MindSpore Sequential Transformer over the 5 epochs of training to check convergence.

In [ ]:
# print("tf loss shape:", len(tf_metrics['loss_history']))
epochs_range = range(1, len(tf_metrics['loss_history']) + 1)
plt.figure(figsize=(7, 4))
plt.plot(epochs_range, tf_metrics['loss_history'], marker='o', color='purple', linewidth=2)
plt.title('Transformer Training Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.xticks(epochs_range)
plt.grid(True, linestyle='--', alpha=0.5)
# plt.savefig('tf_loss.png')
plt.show()


### 4.2 Hyperparameter Grid Search for Sequential Transformer
Let's run a Grid Search over Sequential Transformer hyperparameters (embedding dimensions and learning rates) to optimize validation loss.

In [ ]:
from src.transformer_model import grid_search_transformer
# Perform grid search on a subsample of user sequence history
best_tf_params = grid_search_transformer()


## 5. Model Evaluation and Comparison

Let's load the results of all models and compare their ROC AUC, Mean F1-score, and training speeds.
We also evaluate the Matrix Factorization model directly on the validation candidates to get a baseline.


In [ ]:
from src.train_all import evaluate_all_models

# Run unified evaluation for all models on validation candidate set
df_results = evaluate_all_models()
df_results


### Visualization of Comparison
Let's plot the comparative performance of the three models.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot ROC AUC
sns.barplot(x='Model', y='ROC AUC', data=df_results, ax=axes[0], palette='Blues_r')
axes[0].set_title('Validation ROC AUC comparison')
axes[0].set_ylim(0.4, 0.9)
axes[0].set_ylabel('ROC AUC')
axes[0].set_xticklabels(df_results['Model'], rotation=15)

# Plot F1-score
sns.barplot(x='Model', y='F1@K', data=df_results, ax=axes[1], palette='Oranges_r')
axes[1].set_title('Validation Mean F1-Score comparison')
axes[1].set_ylim(0.0, 0.45)
axes[1].set_ylabel('Mean F1-Score')
axes[1].set_xticklabels(df_results['Model'], rotation=15)

plt.tight_layout()
plt.show()


### Unified Recommendation Metrics Comparison
Let's also compare the models on the newly introduced recommendation metrics: Precision@10, Recall@10, Hit Rate@10, NDCG@10, and MRR.

In [ ]:
# Plot all recommendation metrics in a 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

rec_cols = ['Precision@K', 'Recall@K', 'Hit Rate@K', 'NDCG@K', 'MRR']
palettes = ['Purples_r', 'Greens_r', 'Reds_r', 'YlOrBr_r', 'GnBu_r']

for idx, col in enumerate(rec_cols):
    sns.barplot(x='Model', y=col, data=df_results, ax=axes[idx], palette=palettes[idx])
    axes[idx].set_title(f'Validation {col} comparison')
    axes[idx].set_ylabel(col)
    axes[idx].set_xticklabels(df_results['Model'], rotation=15)
    max_val = df_results[col].astype(float).max()
    axes[idx].set_ylim(0.0, min(1.0, max_val * 1.2))

# Hide the 6th empty subplot
axes[-1].axis('off')

plt.tight_layout()
plt.show()


### Feature Importances in XGBoost
Let's see which features were most important to our best model (XGBoost).


In [ ]:
importances = xgb_metrics['feature_importances']
df_imp = pd.DataFrame(list(importances.items()), columns=['Feature', 'Importance']).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=df_imp, palette='viridis')
plt.title('XGBoost Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


## 6. Interactive Sandbox Demo (Gradio Frontend)
Let's launch an interactive Gradio web application directly in the notebook! This sandbox allows you to:
1. **Select a User ID** to view their purchase history.
2. **Add items to a custom cart** (build your own shopping sequence).
3. **Predict next purchases** and compare the recommendations from Matrix Factorization, XGBoost, and the Sequential Transformer side-by-side.

In [20]:
# Launch Gradio app
from src.gradio_app import demo
# To run inline in Jupyter, set inline=True. height=1000 provides enough spacing.
demo.launch(inline=True, quiet=True, height=1000)
